# DenseGCN Artifact Analysis Notebook

This notebook reads training artifacts produced by `src/train_dense_gcn.py` and generates:
- 8-metric fold plots
- fold/mean/std result tables
- runtime and RAM usage table
- report-ready text snippets for section II-A.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

notebook_dir = Path('.').resolve()
repo_root = notebook_dir.parent
sys.path.insert(0, str(repo_root))

from utils.metrics import METRIC_ORDER
from utils.plotting import plot_folds

pd.set_option('display.max_columns', 50)

## Select Artifact Directory

Examples:
- `artifacts/dense_gcn_v2`
- `artifacts/dense_gcn_v3`

In [ ]:
ARTIFACT_DIR = repo_root / 'artifacts' / 'dense_gcn_v3'

CV_SUMMARY_PATH = ARTIFACT_DIR / 'cv_summary.json'
RESOURCE_PATH = ARTIFACT_DIR / 'resource_summary.json'
FULL_PATH = ARTIFACT_DIR / 'full_retrain_summary.json'

print('Using artifacts from:', ARTIFACT_DIR)
print('CV summary exists:      ', CV_SUMMARY_PATH.exists())
print('Resource summary exists:', RESOURCE_PATH.exists())
print('Full summary exists:    ', FULL_PATH.exists())

In [ ]:
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

cv_summary = load_json(CV_SUMMARY_PATH)
resource_summary = load_json(RESOURCE_PATH)
full_summary = load_json(FULL_PATH) if FULL_PATH.exists() else None

print('Task:', cv_summary.get('task'))
print('Device:', cv_summary.get('device'))

## 8-Metric CV Plots

In [ ]:
fold_metrics = [fold['metrics'] for fold in cv_summary['folds']]
plot_folds(fold_metrics, split_if_obscured=True, verbose=True)

## Fold Results Table (for Report II-A.b)

In [ ]:
rows = []
for fold in cv_summary['folds']:
    row = {
        'Fold': f"Fold {fold['fold']}",
        'Best val loss': fold['best_val_loss'],
        'Best epoch': fold['best_epoch'],
        'Fold time (min)': fold['fold_seconds'] / 60.0,
    }
    for k in METRIC_ORDER:
        row[k] = fold['metrics'][k]
    rows.append(row)

df_folds = pd.DataFrame(rows)
display(df_folds)

mean_row = {'Fold': 'Mean', 'Best val loss': np.nan, 'Best epoch': np.nan, 'Fold time (min)': np.nan}
std_row = {'Fold': 'Std', 'Best val loss': np.nan, 'Best epoch': np.nan, 'Fold time (min)': np.nan}
for k in METRIC_ORDER:
    vals = df_folds[k].astype(float).values
    mean_row[k] = float(np.nanmean(vals))
    std_row[k] = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0

df_stats = pd.DataFrame([mean_row, std_row])
display(df_stats)

## Runtime + RAM Table (for Report II-A.c)

In [ ]:
df_resource = pd.DataFrame([
    {
        'Device': resource_summary.get('device'),
        'Total CV time (sec)': resource_summary.get('total_cv_seconds'),
        'Total CV time (min)': resource_summary.get('total_cv_minutes'),
        'Peak RAM RSS (GB)': resource_summary.get('peak_ram_gb_rss'),
        'psutil available': resource_summary.get('psutil_available'),
    }
])
display(df_resource)

if full_summary is not None:
    df_full = pd.DataFrame([
        {
            'Task': full_summary.get('task'),
            'Train time (sec)': full_summary.get('train_seconds'),
            'Train time (min)': full_summary.get('train_minutes'),
            'Peak RAM RSS (GB)': full_summary.get('peak_ram_gb_rss'),
            'Submission path': full_summary.get('submission_path'),
            'Rows': full_summary.get('submission_rows'),
        }
    ])
    display(df_full)

## Report Snippet Generator (II-A)

Fill Kaggle score/rank manually after submission.

In [ ]:
mean_map = cv_summary['mean_metrics']
std_map = cv_summary['std_metrics']

def pm(metric_name: str) -> str:
    return f"{mean_map[metric_name]:.6f} +/- {std_map[metric_name]:.6f}"

text_b = (
    'Across 3-fold cross-validation, the model achieved '\
    f"MAE={pm('MAE')}, PCC={pm('PCC')}, JSD={pm('JSD')}, "
    f"PC MAE={pm('MAE (PC)')}, EC MAE={pm('MAE (EC)')}, BC MAE={pm('MAE (BC)')}, "
    f"Strength MAE={pm('MAE (Strength)')}, and Clustering MAE={pm('MAE (Clustering)')}."
)

text_c = (
    f"The total 3-fold training time was {resource_summary['total_cv_minutes']:.2f} minutes "
    f"({resource_summary['total_cv_seconds']:.1f} seconds), with peak RAM usage of "
    f"{resource_summary['peak_ram_gb_rss']} GB on {resource_summary['device']}."
)

print('II-A(b) draft:')
print(text_b)
print('\nII-A(c) draft:')
print(text_c)

print('\nII-A(d) template:')
print(
    'After selecting hyperparameters from 3-fold CV, we retrained on the full train_LR/train_HR set and '
    'predicted test_HR from test_LR to create the final Kaggle submission. '
    'Public score: <fill>, Rank: <fill>/<fill>.'
)